# Hybrid Analytics System Dashboard
MongoDB + Cassandra Analytics

In [18]:
import pandas as pd
import plotly.express as px
from pymongo import MongoClient
from cassandra.cluster import Cluster

In [19]:
mongo = MongoClient('mongodb://localhost:27017')
mongo_db = mongo.hybrid_analytics
products = mongo_db.products

cluster = Cluster(['localhost'])
session = cluster.connect('hybrid_analytics')

print('Connected Successfully')

Connected Successfully


## Products by Category

In [20]:
pipeline = [{'$group': {'_id': '$category', 'count': {'$sum': 1}}}]
category_df = pd.DataFrame(list(products.aggregate(pipeline)))
category_df.columns = ['category','count']
fig = px.bar(
    category_df,
    x='category',
    y='count',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Category'
)
fig.show()

## Products by Brand

In [21]:
pipeline = [{'$group': {'_id': '$brand', 'count': {'$sum': 1}}}]
brand_df = pd.DataFrame(list(products.aggregate(pipeline)))
brand_df.columns = ['brand','count']
fig = px.bar(
    brand_df.sort_values('count'),
    x='count',
    y='brand',
    color='brand',
    orientation='h',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

## Price Distribution

In [22]:
prices = pd.DataFrame(list(products.find({}, {'price': 1})))
fig = px.histogram(prices, x='price', nbins=30, title='Price Distribution Histogram')
fig.show()


## Average Price by Category

In [23]:
pipeline = [{'$group': {'_id': '$category', 'avg_price': {'$avg': '$price'}}}]
avg_df = pd.DataFrame(list(products.aggregate(pipeline)))
avg_df.columns = ['category','avg_price']

fig = px.bar(
    avg_df,
    x='category',
    y='avg_price',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Average Price by Category'
)
fig.update_layout(showlegend=False)
fig.show()

## Cassandra Metrics

In [24]:
metric_rows = session.execute('SELECT * FROM product_metrics')
metrics_df = pd.DataFrame(list(metric_rows))
metrics_df.head()

,product_id,event_type,count
0,23,view,2
1,114,view,1
2,110,view,3
3,91,view,1
4,128,favorite,2


In [25]:
event_rows = session.execute('SELECT * FROM product_events')
event_df = pd.DataFrame(list(event_rows))
event_df.head()

,product_id,event_type,event_time
0,23,view,2026-07-22 23:42:20.372
1,23,view,2026-07-22 23:42:20.043
2,114,view,2026-07-22 23:42:21.301
3,110,view,2026-07-22 23:42:21.493
4,110,view,2026-07-22 23:42:21.121


## Top Viewed Products

In [26]:
if 'product_df' not in globals():
    product_df = pd.DataFrame(list(products.find()))
    if 'id' not in product_df.columns and '_id' in product_df.columns:
        try:
            product_df['id'] = product_df['_id'].astype(int)
        except Exception:
            product_df['id'] = product_df['_id'].astype(str)

In [27]:
metrics_df


,product_id,event_type,count
0,23,view,2
1,114,view,1
2,110,view,3
3,91,view,1
4,128,favorite,2
...,...,...,...
253,3,view,1
254,161,favorite,2
255,161,view,4
256,103,favorite,3


In [28]:
views_df = metrics_df[metrics_df['event_type'] == 'view'].sort_values('count', ascending=False).head(10)

views_products = views_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)

views_products['label'] = views_products['title'] + ' (' + views_products['brand'] + ')'

fig = px.pie(
    views_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Viewed Products'
)
fig.show()

## Top Favorited Products

In [29]:
fav_df = metrics_df[metrics_df['event_type'] == 'favorite'].sort_values('count', ascending=False).head(10)
fav_products = fav_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
fav_products['label'] = fav_products['title'] + ' (' + fav_products['brand'] + ')'

fig = px.pie(
    fav_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Favorited Products'
)
fig.show()

## Top Purchased Products

In [30]:
buy_df = metrics_df[metrics_df['event_type'] == 'buy'].sort_values('count', ascending=False).head(10)
buy_products = buy_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
buy_products['label'] = buy_products['title'] + ' (' + buy_products['brand'] + ')'

fig = px.pie(
    buy_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Purchased Products'
)
fig.show()

## Product Engagement Score

In [31]:
fig = px.bar(
    brand_df,
    x='brand',
    y='count',
    color='brand',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

In [32]:
metrics_pivot = (
    metrics_df
    .pivot(index='product_id', columns='event_type', values='count')
    .reset_index()
    .fillna(0)
)
metrics_pivot.columns.name = None
metrics_pivot = metrics_pivot.astype({'buy': int, 'favorite': int, 'view': int})

metrics_pivot = metrics_pivot.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
metrics_pivot.head()

,product_id,buy,favorite,view,id,title,brand
0,1,0,1,2,1,Cross-platform executive algorithm,Philips
1,2,1,0,2,2,Enterprise-wide 6thgeneration framework,Dell
2,3,0,0,1,3,Ergonomic optimizing synergy,Apple
3,4,1,0,1,4,Customizable composite portal,Puma
4,5,0,1,0,5,Digitized multi-tasking structure,Apple


In [33]:
metric_rows = session.execute('SELECT * FROM product_metrics')
metrics_df = pd.DataFrame(list(metric_rows))
metrics_df = metrics_df.sort_values('product_id')
metrics_df.head(10)

,product_id,event_type,count
64,1,favorite,1
65,1,view,2
79,2,view,2
78,2,buy,1
253,3,view,1
86,4,view,1
85,4,buy,1
15,5,favorite,1
160,6,view,1
143,7,view,2


In [34]:

event_rows = session.execute('SELECT * FROM product_events')
event_df = pd.DataFrame(list(event_rows))
event_df = event_df.sort_values('event_time', ascending=False)
print("Total Events:", len(event_df))
event_df.head()

Total Events: 500


,product_id,event_type,event_time
345,108,view,2026-07-22 23:42:21.775
451,192,view,2026-07-22 23:42:21.771
154,45,view,2026-07-22 23:42:21.767
249,189,view,2026-07-22 23:42:21.763
103,13,view,2026-07-22 23:42:21.760
